# **Full Pipeline**

In [5]:
from pathlib import Path
import sys, os, platform

CWD  = Path.cwd().resolve()
ROOT = CWD if (CWD / "src").exists() else CWD.parent
if str(ROOT) not in sys.path: sys.path.append(str(ROOT))

In [ ]:
DOC_ID = "New_York_State_Workers_Compensation_Medical_Fee"   # e.g., "title17"
PDF = ROOT / "data" / "raw" / f"{DOC_ID}.pdf"
RUN_DIR = ROOT / "data" / "runs" / DOC_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

MD_DIR     = RUN_DIR / "md"
CHUNKS_DIR = RUN_DIR / "chunks"
INDEX_DIR  = RUN_DIR / "index"
SFT_DIR    = RUN_DIR / "sft"
PAIRS_DIR  = RUN_DIR / "pairs"

print(f"[run] DOC_ID={DOC_ID}")
print(f"[pdf] {PDF}")
print(f"[out] {RUN_DIR}")

[run] DOC_ID=New_York_State_Workers_Compensation_Medical_Fee
[pdf] D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\raw\New_York_State_Workers_Compensation_Medical_Fee.pdf
[out] D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee


### **1) PDF to MD**

In [11]:
from src.ingest.pdf_to_markdown import convert_pdf_to_markdown
paths = convert_pdf_to_markdown(PDF, MD_DIR, ocr=True)
print("markdown pages:", paths)

markdown pages: {'markdown': WindowsPath('D:/IIT BBS/Job Resources/Business Optima/pdf-agent/data/runs/New_York_State_Workers_Compensation_Medical_Fee/md/New_York_State_Workers_Compensation_Medical_Fee.md'), 'toc_json': WindowsPath('D:/IIT BBS/Job Resources/Business Optima/pdf-agent/data/runs/New_York_State_Workers_Compensation_Medical_Fee/md/New_York_State_Workers_Compensation_Medical_Fee.toc.json'), 'pages_jsonl': WindowsPath('D:/IIT BBS/Job Resources/Business Optima/pdf-agent/data/runs/New_York_State_Workers_Compensation_Medical_Fee/md/New_York_State_Workers_Compensation_Medical_Fee.pages.jsonl'), 'images_dir': None}


### **2) Chunking**

In [12]:
from src.ingest.md_to_chunks import md_to_chunks
CHUNKS = CHUNKS_DIR / "chunks.jsonl"
n = md_to_chunks(
    MD_DIR / f"{DOC_ID}.md", CHUNKS,
    pages_jsonl=MD_DIR / f"{DOC_ID}.pages.jsonl",
    max_chars=1600, overlap=400, drop_gibberish=True, drop_toc=False
)
print("chunks:", n)

chunks: 1494


### **3) Index Building**

In [ ]:
from src.ingest.build_index import build_index
collection_name = build_index(
    CHUNKS, persist=INDEX_DIR,
    embed_model="BAAI/bge-base-en-v1.5",
    batch_size=64, bge_use_prompt=True, reset=True
)
print("collection:", collection_name)

Indexed 64/1494
Indexed 128/1494
Indexed 192/1494
Indexed 256/1494


### **4) SFT Data Gen + Cleaning**

In [ ]:
from src.ingest.make_qa_and_summaries import make_data
summaries_path, qa_path = make_data(
    chunks_path=CHUNKS,
    out_dir=SFT_DIR,
    model="llama3:instruct", # llama3.1:8b-instruct-q8_0
    max_sections=6, min_tokens_per_section=600,
    qa_per_section=3
)
# run sanitize_summaries_v3b + clean_qa_v3b (reuse your functions)

### **5) LoRA/PEFT Training**

In [ ]:
OUT = ROOT / "outputs" / "lora_hf" / DOC_ID
OUT.mkdir(parents=True, exist_ok=True)
# call cpu_lora_hf.py with train.jsonl in SFT_DIR

### **6) Reranker Training**

In [ ]:
OUT_RR = ROOT / "outputs" / "reranker" / DOC_ID
OUT_RR.mkdir(parents=True, exist_ok=True)
# call retriever_pairs.py + train_reranker.py